## Processing of Moving Vessel Profiler Data - code 7_apply_final_offset_leg1
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es)
### Data from FaSt-SWOT experiment

**INPUT**: step5*.nc files (no offset applied)

**OUTPUT**: step6_with_offset/*.nc files (with final offset applied)

The offset is calculated from CTD vs MVP comparison in `6_offset_computation_leg1.ipynb`

In [1]:
import xarray as xr
import numpy as np
import gsw
from pathlib import Path

print('Applying final offset to step5 datasets...')

# ==========================================
# CONFIGURATION
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")
DIR_IN = BASE_ROOT / "Data" / "Leg1" / "processed_step5"
DIR_OUT = BASE_ROOT / "Data" / "Leg1" / "processed_step6_with_offset"
DIR_OUT.mkdir(parents=True, exist_ok=True)

# FINAL OFFSET calculated from CTD comparison
# From final_offset_and_figure_leg1.ipynb analysis
FINAL_OFFSET = -0.03  # PSU (median offset from CTD-MVP comparison)

print(f'Applying offset: {FINAL_OFFSET} PSU')
print(f'Input directory: {DIR_IN}')
print(f'Output directory: {DIR_OUT}')

# ==========================================
# PROCESSING
# ==========================================
files = sorted(list(DIR_IN.glob('*_step5.nc')))

if not files:
    print('ERROR: No step5 files found!')
else:
    count_ok = 0
    count_error = 0
    
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                p = ds['pressure'].values
                t_final = ds['t1_smooth'].values
                s_final = ds['salinity_practical'].values  # NO offset yet
                lat = ds['latitude'].values
                lon = ds['longitude'].values

                # Apply offset
                s_with_offset = s_final + FINAL_OFFSET
                
                # Recalculate TEOS-10 variables with offset
                SA = gsw.SA_from_SP(s_with_offset, p, lon, lat)  # Absolute Salinity
                CT = gsw.CT_from_t(SA, t_final, p)  # Conservative Temperature
                sigma0 = gsw.sigma0(SA, CT)  # Potential Density
                
                # Recalculate conductivity consistent with offset salinity
                c_with_offset = gsw.C_from_SP(s_with_offset, t_final, p)

                # Create output dataset - ONLY 4 main variables + coordinates
                coords_dict = {
                    'scan': ds['scan'].values,
                }
                
                # Add pressure as array coordinate (varies with scan)
                if p.ndim == 0:  # scalar
                    coords_dict['pressure'] = float(p)
                else:
                    coords_dict['pressure'] = (('scan',), p)
                
                ds_out = xr.Dataset(
                    data_vars={
                        'conservative_temperature_processed': (('scan',), CT),
                        'practical_salinity_processed': (('scan',), s_with_offset),
                        'conductivity_processed': (('scan',), c_with_offset),
                        'sigma_theta': (('scan',), sigma0)
                    },
                    coords=coords_dict
                )
                
                # Add scalar coordinates (latitude, longitude, time)
                if lat.ndim == 0:  # scalar
                    ds_out = ds_out.assign_coords(latitude=float(lat))
                else:  # array
                    ds_out.coords['latitude'] = (('scan',), lat)
                    
                if lon.ndim == 0:  # scalar
                    ds_out = ds_out.assign_coords(longitude=float(lon))
                else:  # array
                    ds_out.coords['longitude'] = (('scan',), lon)
                
                if 'time' in ds.coords:
                    ds_out = ds_out.assign_coords(time=ds['time'].values)
                
                # Update metadata
                ds_out['practical_salinity_processed'].attrs = {
                    'units': 'PSU',
                    'standard_name': 'sea_water_practical_salinity',
                    'comment': f'Final processed salinity with offset {FINAL_OFFSET} PSU applied (CTD calibration). Result of QC, lag correction, smoothing, and TEOS-10 conversion.'
                }
                ds_out['conservative_temperature_processed'].attrs = {
                    'units': 'degC',
                    'standard_name': 'sea_water_conservative_temperature',
                    'comment': 'Final processed temperature (TEOS-10 CT). Result of QC, lag correction, smoothing, and thermodynamic calculation.'
                }
                ds_out['conductivity_processed'].attrs = {
                    'units': 'mS/cm',
                    'standard_name': 'sea_water_electrical_conductivity',
                    'comment': 'Final processed conductivity recalculated from practical_salinity_processed and conservative_temperature_processed.'
                }
                ds_out['sigma_theta'].attrs = {
                    'units': 'kg/m^3',
                    'standard_name': 'sea_water_sigma_theta',
                    'comment': 'Potential density anomaly calculated from processed salinity and temperature (TEOS-10).'
                }

                # Global metadata update for publication-ready NetCDF
                ds_out.attrs['Conventions'] = 'CF-1.6'
                ds_out.attrs['processing_level'] = 'Step 6 - final offset applied'
                ds_out.attrs['offset_applied_psu'] = float(FINAL_OFFSET)
                ds_out.attrs['processed_by'] = 'Elisabet Verger-Miralles'
                ds_out.attrs['processing_code_repository'] = 'https://github.com/everger-miralles/MVPytools'
                ds_out.attrs['ship'] = 'R/V SOCIB'
                ds_out.attrs['cruise'] = 'FaSt-SWOT Leg 1'

                if 'time' in ds.coords:
                    tval = str(np.atleast_1d(ds['time'].values).flatten()[0])
                    ds_out.attrs['date_utc'] = tval[:10] if len(tval) >= 10 else 'unknown'
                    ds_out.attrs['time_utc'] = tval[11:19] if len(tval) >= 19 else 'unknown'
                else:
                    ds_out.attrs['date_utc'] = 'unknown'
                    ds_out.attrs['time_utc'] = 'unknown'

                ds_out.attrs['history'] = f'Step 6 final offset applied ({FINAL_OFFSET} PSU). Final processed variables: conservative_temperature_processed, practical_salinity_processed, conductivity_processed, sigma_theta. Complete processing pipeline: QC → lag correction → smoothing → TEOS-10 conversion → offset application.'

                # Save
                out_name = f.name.replace('_step5.nc', '_step6_with_offset.nc')
                ds_out.to_netcdf(DIR_OUT / out_name)
                count_ok += 1
            
        except Exception as e:
            print(f'  ERROR in {f.name}: {e}')
            count_error += 1

    print(f"\n{'='*70}")
    print(f"Process completed!")
    print(f"Successfully processed: {count_ok} files")
    if count_error > 0:
        print(f"Errors: {count_error}")
    print(f"Files saved in: {DIR_OUT}")
    print(f"Final variables: conservative_temperature_processed, practical_salinity_processed, conductivity_processed, sigma_theta")
    print(f"{'='*70}")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Applying final offset to step5 datasets...
Applying offset: -0.03 PSU
Input directory: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg1\processed_step5
Output directory: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg1\processed_step6_with_offset

Process completed!
Successfully processed: 591 files
Files saved in: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg1\processed_step6_with_offset
Final variables: conservative_temperature_processed, practical_salinity_processed, conductivity_processed, sigma_theta
